In [ ]:
## 1. Configuração

import requests
import json
import time
import pandas as pd

In [26]:
## 2. Exploração da API

# Escolhe qualquer username público do Lichess
USERNAME = "DrNykterstein"  # Magnus Carlsen no Lichess
url = f"https://lichess.org/api/games/user/{USERNAME}"

# Requisição com análise completa do Stockfish
params = {
    "rated":    "true",
    "max":      1,
    "opening":  "true",
    "evals":    "true",    # avaliações do Stockfish por lance
    "clocks":   "true",    # tempo no relógio por lance
    "analysed": "true",    # só partidas já analisadas
}
headers = {"Accept": "application/x-ndjson"}

response = requests.get(url, params=params, headers=headers)
game = json.loads(response.text)

# Inspeciona estrutura
print("CLOCKS:", game["clocks"][:5])
print("ANALYSIS:", game["analysis"][:5])

# Lances com julgamento do Stockfish
erros_exemplo = [l for l in game["analysis"] if "judgment" in l]
print(f"\nLances com erro: {len(erros_exemplo)}")
print(erros_exemplo[0])

CLOCKS: [18003, 18003, 17915, 17907, 17891]
ANALYSIS: [{'eval': 18}, {'eval': 68}, {'eval': 53}, {'eval': 51}, {'eval': 10}]

Lances com erro: 14
{'eval': 119, 'best': 'e7e6', 'variation': 'e6 Qe2 c5 Rd1 Qd7 Qe4 d5 Qg4 f5 exf6 gxf6 c4', 'judgment': {'name': 'Inaccuracy', 'comment': 'Inaccuracy. e6 was best.'}}


In [27]:
## 3. Extração de Erros

def extrair_erros(game):
    erros = []
    
    analise  = game.get("analysis", [])
    relogios = game.get("clocks", [])
    
    speed        = game.get("speed", "unknown")
    abertura     = game.get("opening", {}).get("name", "Desconhecida")
    rating_white = game["players"]["white"].get("rating")
    rating_black = game["players"]["black"].get("rating")
    
    for i, lance in enumerate(analise):
        if "judgment" not in lance:
            continue  # lance sem erro, pula
        
        cor           = "white" if i % 2 == 0 else "black"
        numero_lance  = (i // 2) + 1
        rating_jogador = rating_white if cor == "white" else rating_black
        relogio        = relogios[i] / 100 if i < len(relogios) else None  # converte para segundos
        
        # Fase do jogo
        if numero_lance <= 15:
            fase = "Abertura"
        elif numero_lance <= 35:
            fase = "Meio-jogo"
        else:
            fase = "Final"
        
        erros.append({
            "game_id":        game["id"],
            "speed":          speed,
            "numero_lance":   numero_lance,
            "fase":           fase,
            "cor":            cor,
            "rating_jogador": rating_jogador,
            "relogio_seg":    relogio,
            "tipo_erro":      lance["judgment"]["name"],  # Inaccuracy / Mistake / Blunder
            "eval":           lance.get("eval"),
            "abertura":       abertura,
        })
    
    return erros

# Testa com a partida que já temos
resultado = extrair_erros(game)
df_teste = pd.DataFrame(resultado)
print(df_teste)

     game_id  speed  numero_lance       fase    cor  rating_jogador  \
0   kAdOQKeh  blitz             8   Abertura  black            3145   
1   kAdOQKeh  blitz            12   Abertura  black            3145   
2   kAdOQKeh  blitz            29  Meio-jogo  white            2644   
3   kAdOQKeh  blitz            32  Meio-jogo  white            2644   
4   kAdOQKeh  blitz            33  Meio-jogo  black            3145   
5   kAdOQKeh  blitz            35  Meio-jogo  black            3145   
6   kAdOQKeh  blitz            47      Final  white            2644   
7   kAdOQKeh  blitz            47      Final  black            3145   
8   kAdOQKeh  blitz            49      Final  white            2644   
9   kAdOQKeh  blitz            51      Final  white            2644   
10  kAdOQKeh  blitz            51      Final  black            3145   
11  kAdOQKeh  blitz            52      Final  white            2644   
12  kAdOQKeh  blitz            58      Final  white            2644   
13  kA

In [28]:
## 4. Coleta em Escala

def coletar_usuarios_aleatorios(n_usuarios=500):
    """
    Coleta usuários aleatórios via torneios abertos do Lichess.
    Sem filtro de rating — reflete a distribuição real da plataforma.
    """
    usernames = set()

    print("Buscando torneios abertos...")
    url = "https://lichess.org/api/tournament"
    r = requests.get(url, headers={"Accept": "application/json"}, timeout=10)
    data = r.json()

    torneios = (
        data.get("created", []) +
        data.get("started", []) +
        data.get("finished", [])
    )

    torneios_abertos = [
        t for t in torneios
        if not t.get("conditions", {}).get("maxRating")
        and not t.get("conditions", {}).get("minRating")
    ]
    print(f"  {len(torneios_abertos)} torneios abertos encontrados")

    for torneio in torneios_abertos:
        if len(usernames) >= n_usuarios:
            break
        tid = torneio.get("id")
        try:
            url2 = f"https://lichess.org/api/tournament/{tid}/results"
            r2 = requests.get(url2, headers={"Accept": "application/x-ndjson"}, timeout=10)
            for line in r2.text.strip().split("\n"):
                if not line:
                    continue
                d = json.loads(line)
                if "username" in d:
                    usernames.add(d["username"])
            print(f"  torneio {tid}: {len(usernames)} usuários acumulados")
            time.sleep(0.8)
        except:
            continue

    return list(usernames)[:n_usuarios]


def coletar_partidas_e_erros(usernames, partidas_por_usuario=20):
    """
    Para cada usuário:
      1. Busca partidas analisadas (todos os modos de jogo)
      2. Extrai os erros usando extrair_erros()
    """
    todos_erros = []

    for username in usernames:
        try:
            url = f"https://lichess.org/api/games/user/{username}"
            params = {
                "max":      partidas_por_usuario,
                "rated":    "true",
                "analysed": "true",
                "evals":    "true",
                "clocks":   "true",
                "opening":  "true",
            }
            r = requests.get(
                url, params=params,
                headers={"Accept": "application/x-ndjson"},
                timeout=15
            )
            for line in r.text.strip().split("\n"):
                if not line:
                    continue
                try:
                    game = json.loads(line)
                    todos_erros.extend(extrair_erros(game))
                except:
                    continue
            time.sleep(1.5)
        except Exception as ex:
            print(f"  erro em {username}: {ex}")
            continue

    return pd.DataFrame(todos_erros)


usuarios = coletar_usuarios_aleatorios(n_usuarios=500)
print(f"\n{len(usuarios)} usuários coletados")

print("\nColetando partidas... (pode levar alguns minutos)")
df = coletar_partidas_e_erros(usuarios, partidas_por_usuario=20)
print(f"  → {len(df):,} erros extraídos")

Buscando torneios abertos...
  139 torneios abertos encontrados
  torneio vdfLtF8L: 3 usuários acumulados
  torneio lTpCexrj: 7 usuários acumulados
  torneio JeCnAovC: 7 usuários acumulados
  torneio cJpSX3Ax: 7 usuários acumulados
  torneio qzuYdRAi: 7 usuários acumulados
  torneio pTEiYLaC: 7 usuários acumulados
  torneio Ht53aj1Y: 7 usuários acumulados
  torneio mrqyl7Xo: 7 usuários acumulados
  torneio 51wj6i4j: 7 usuários acumulados
  torneio GyHyHpxq: 9 usuários acumulados
  torneio NpyVi5lk: 9 usuários acumulados
  torneio bQYtEbAU: 9 usuários acumulados
  torneio w4MK2e0U: 13 usuários acumulados
  torneio SfWgYzDE: 13 usuários acumulados
  torneio IyMU4F7G: 13 usuários acumulados
  torneio KbdFz60U: 13 usuários acumulados
  torneio pcv52H7x: 13 usuários acumulados
  torneio y48mc25J: 14 usuários acumulados
  torneio 9d0WQyfd: 15 usuários acumulados
  torneio P9Mj6Xmv: 15 usuários acumulados
  torneio KAtf6mPt: 16 usuários acumulados
  torneio IwNjdCMF: 16 usuários acumulados
  

In [29]:
## 5. Tratamento e Validação

def classificar_faixa(rating):
    if rating is None:
        return None
    elif rating < 1000:
        return "Iniciante (400-1000)"
    elif rating < 1600:
        return "Intermediário (1000-1600)"
    elif rating < 2200:
        return "Avançado (1600-2200)"
    else:
        return "Expert (2200+)"

df["faixa"] = df["rating_jogador"].apply(classificar_faixa)
df = df[df["faixa"].notna()].copy()

print("Distribuição por faixa:")
print(df["faixa"].value_counts())

print("\nRatings por faixa:")
print(df.groupby("faixa")["rating_jogador"].describe()[["min", "mean", "max"]])

print("\nDistribuição por modalidade:")
print(df["speed"].value_counts())

print("\nDistribuição por tipo de erro:")
print(df["tipo_erro"].value_counts())

Distribuição por faixa:
faixa
Avançado (1600-2200)         86664
Intermediário (1000-1600)    44399
Expert (2200+)               17143
Iniciante (400-1000)          3932
Name: count, dtype: int64

Ratings por faixa:
                              min         mean     max
faixa                                                 
Avançado (1600-2200)       1600.0  1880.111269  2199.0
Expert (2200+)             2200.0  2356.172432  3165.0
Iniciante (400-1000)        457.0   886.816124   999.0
Intermediário (1000-1600)  1000.0  1377.660420  1599.0

Distribuição por modalidade:
speed
blitz             65746
bullet            44949
ultraBullet       24840
rapid             14896
classical          1562
correspondence      145
Name: count, dtype: int64

Distribuição por tipo de erro:
tipo_erro
Blunder       61970
Inaccuracy    60700
Mistake       29468
Name: count, dtype: int64


In [30]:
## 6. Exportação

output_path = "../data/blunders.csv"
df.to_csv(output_path, index=False)

print(f"✅ CSV exportado: {output_path}")
print(f"   Linhas: {len(df):,}")
print(f"   Colunas: {list(df.columns)}")
df.head()

✅ CSV exportado: ../data/blunders.csv
   Linhas: 152,138
   Colunas: ['game_id', 'speed', 'numero_lance', 'fase', 'cor', 'rating_jogador', 'relogio_seg', 'tipo_erro', 'eval', 'abertura', 'faixa']


,game_id,speed,numero_lance,fase,cor,rating_jogador,relogio_seg,tipo_erro,eval,abertura,faixa
0,2yg1eCMf,blitz,3,Abertura,white,1848,182.27,Blunder,-237.0,Alekhine Defense,Avançado (1600-2200)
1,2yg1eCMf,blitz,6,Abertura,white,1848,157.79,Blunder,-630.0,Alekhine Defense,Avançado (1600-2200)
2,2yg1eCMf,blitz,7,Abertura,black,2303,163.23,Blunder,-130.0,Alekhine Defense,Expert (2200+)
3,2yg1eCMf,blitz,9,Abertura,white,1848,154.91,Blunder,-615.0,Alekhine Defense,Avançado (1600-2200)
4,2yg1eCMf,blitz,9,Abertura,black,2303,153.23,Blunder,-284.0,Alekhine Defense,Expert (2200+)
